In [ ]:
import os, time, glob
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

import glob
import igraph as ig

import numpy as np
from pathlib import Path


ROOT = "data"

# Cosine similarity output folder
SIM_DIR = os.path.join(ROOT, "cosine_matrices")
os.makedirs(SIM_DIR, exist_ok=True)

#igraph output folder 
G_DIR = os.path.join(ROOT, "igraph_matrices")
os.makedirs(G_DIR, exist_ok=True)

# Cosine Similarity Matrix Computation

In [ ]:
def load_vec_npz(path):
    d = np.load(path, allow_pickle=True)
    words = np.array(d["words"], dtype=object)
    vectors = np.asarray(d["vectors"])
    if vectors.ndim != 2 or len(words) != vectors.shape[0]:
        raise ValueError("words/vectors shape mismatch.")
    return words, vectors


def compute_and_save_cosine_npz(in_npz_path):
    t0 = time.time()
    words, vectors = load_vec_npz(in_npz_path)
    sim = cosine_similarity(vectors).astype(np.float32, copy=False)
    stem = Path(in_npz_path).stem
    out_path = os.path.join(SIM_DIR, f"{stem}.cosine_sim.npz")
    np.savez_compressed(out_path, sim=sim, words=words)
    print(f"[OK] {stem}: sim={sim.shape} saved to {out_path} in {time.time()-t0:.2f}s")

pattern = os.path.join(ROOT, "*", "*.top10k.embeddings.simw.npz")
files = sorted(glob.glob(pattern))

print(f"Found {len(files)} language files.")
for f in files:
    compute_and_save_cosine_npz(f)


# Semantic network construction from cosine similarity matrices

This notebook constructs semantic networks from precomputed cosine similarity matrices. Each matrix represents pairwise cosine similarities between the top 10,000 word embeddings of one language.

Two network construction methods are applied:

1. **Mutual k-nearest-neighbor networks**, where edges are retained only when two words occur in each other's top-k nearest-neighbor lists.
2. **Threshold-based networks**, where edges are retained if their cosine similarity is at or above the 95th percentile of all off-diagonal similarity values.

The resulting networks are saved as undirected `igraph` graphs in GraphML format. Word labels are stored as vertex names, and cosine similarity values are stored as edge weights.

# 1.Mutual k-nearest-neighbor network 

In [ ]:
def load_sim_npz(npz_path):
    """
    Load a saved cosine similarity `.npz` file.

    Returns:
        sim: cosine similarity matrix
        words: word labels for matrix rows/columns
    """
    data = np.load(npz_path, allow_pickle=True)
    sim   = np.asarray(data["sim"])
    words = np.array(data["words"], dtype=object)
    if sim.shape[0] != sim.shape[1] or sim.shape[0] != len(words):
        raise ValueError(f"Shape mismatch in {npz_path}: sim={sim.shape}, words={len(words)}")
    if not np.all(np.isfinite(sim)):
        raise ValueError(f"Non-finite values in sim for {npz_path}")
    return sim, words

def mknn_edges_from_sim(sim, k=1000, mutual=True):
    """
    Build a mutual kNN edge list from a cosine similarity matrix.

    - Keeps (i, j) only if i is in j's top-k and j is in i's top-k, if mutual=True.
    - Uses argpartition for efficient top-k selection.
    - Sorts the selected top-k neighbors to recover exact ranks.
    - Returns edges, weights, and both endpoint ranks.

    Returns:
        edges: list[(i, j)] with i < j
        weights: list[float] cosine similarity for each edge
        rank_i: list[int] rank of j in i's top-k list, where 0 = most similar
        rank_j: list[int] rank of i in j's top-k list, where 0 = most similar
    """
    n = sim.shape[0]
    if n < 2:
        return [], [], [], []
    
    # Work on a float32 copy; set self-sim to -inf so it never appears in top-k
    S = sim.astype(np.float32, copy=True)
    np.fill_diagonal(S, -np.inf)

    # Clip k to valid range [1, n-1]
    k = int(max(1, min(k, n - 1)))

    # 1) Fast top-k (unsorted) using argpartition
    idx_topk = np.argpartition(-S, kth=k-1, axis=1)[:, :k]  # (n,k)

    # 2) Exact ordering: sort those k neighbors by descending similarity
    row = np.arange(n)[:, None]
    vals_topk = S[row, idx_topk]               # (n,k) cosine values for chosen neighbors
    order = np.argsort(-vals_topk, axis=1)     # indices to sort descending
    idx_topk = idx_topk[row, order]            # now each row's kNN list is sorted best to worst

    # 3) For mutual check and rank recording
    knn_sets  = [set(idx_topk[i].tolist()) for i in range(n)] 
    rank_maps = [{int(idx_topk[i, r]): r for r in range(k)} for i in range(n)] 

    # 4) Build undirected edge list once (i<j), with weights and both endpoint ranks
    edges, weights, rank_i, rank_j = [], [], [], []
    for i in range(n):
        for j in idx_topk[i]:
            if mutual and i not in knn_sets[j]: # enforcing mutuality
                continue  

            if i < j:  # add each undirected edge only once
                edges.append((int(i), int(j)))
                weights.append(float(S[i, j]))
                rank_i.append(int(rank_maps[i][int(j)]))  # rank of j in i's kNN (0 best)
                rank_j.append(int(rank_maps[j][int(i)]))  # rank of i in j's kNN (0 best)

    return edges, weights, rank_i, rank_j


def build_igraph(n, edges, weights, words, rank_i, rank_j):
    """
    Create an undirected igraph graph with word labels and edge attributes for cosine similarity and ranks.
    """
    g = ig.Graph(n=n, edges=edges, directed=False)

    g.vs["name"] = [str(w) for w in words]
    
    # Store cosine similarity and kNN rank information as edge attributes.
    g.es["weight"] = list(weights)  # igraph prefers lists
    g.es["rank_i"] = list(rank_i)
    g.es["rank_j"] = list(rank_j)
    return g

def save_graph(g, out_stem):
    graphml_path = out_stem + ".graphml"
    g.save(graphml_path)  
    return graphml_path


In [ ]:

pattern = os.path.join(SIM_DIR, "*.cosine_sim.npz")
files = sorted(glob.glob(pattern))
print(f"Found {len(files)} cosine matrices in {SIM_DIR}")

# Build one mutual kNN graph per cosine similarity matrix.
for npz_path in files:
    stem = Path(npz_path).stem  
    out_stem = os.path.join(G_DIR, f"{stem}.mknn100")

    #time the whole process for this file since it can be slow for large matrices
    t0 = time.time() 
    sim, words = load_sim_npz(npz_path)

    #just to check the early stats of the graph before saving
    edges, weights, r_i, r_j = mknn_edges_from_sim(sim, k=100, mutual=True)
    n = len(words) 
    m = len(edges) 
    avg_degree = (2.0 * m) / max(1, n) 

    g = build_igraph(n, edges, weights, words, r_i, r_j)
    out_path = save_graph(g, out_stem)

    print(f"[OK] {stem}: nodes={n} edges={len(edges)} avg_deg≈{avg_degree:.1f} → {out_path} ({time.time()-t0:.2f}s)") 

## 2. Threshold-Based Network

In [ ]:
PERCENTILE = 95 # global threshold percentile

def edges_from_global_percentile(sim, p=95, mutual=True, weighted=True):

    """
    Build threshold-based edges from a cosine similarity matrix.

    The cutoff is the 95th percentile of the off-diagonal similarity values.
    Edges are kept when their cosine similarity is greater than or equal to
    this cutoff.

    Returns:
        edges: undirected edges `(i, j)` with `i < j`
        weights: cosine similarity values for the retained edges
    """
    n = sim.shape[0]
    if n < 2:
        return [], []

    S = sim.astype(np.float32, copy=True)
    np.fill_diagonal(S, -np.inf)

    # Using upper triangle to avoid double-counting
    off = S[np.triu_indices_from(S, k=1)]
    eps = float(np.percentile(off, p))

    # threshold mask
    mask = S >= eps
    if mutual:
        mask = np.logical_and(mask, mask.T)  # robust even if S has tiny asymmetries

    # extract undirected edges once (upper triangle)
    ii, jj = np.where(np.triu(mask, 1))
    if weighted:
        ww = S[ii, jj].astype(np.float32)
        return list(zip(ii.tolist(), jj.tolist())), ww.tolist()
    else:
        return list(zip(ii.tolist(), jj.tolist())), [1.0] * len(ii)
    
#threshold graph with no ranks
    
def build_igraph_threshold(n, edges, weights, words):
    """Build an undirected igraph graph for threshold edges"""
    g = ig.Graph(n=n, edges=edges, directed=False)
    g.vs["name"]   = [str(w) for w in words]
    g.es["weight"] = list(np.asarray(weights, dtype=np.float32))
    return g

    

pattern = os.path.join(SIM_DIR, "*.cosine_sim.npz")
all_thr_files = sorted(glob.glob(pattern))

thr_files = all_thr_files[:N_SAMPLE] if ('N_SAMPLE' in globals() and N_SAMPLE is not None) else all_thr_files
print(f"Threshold(p={PERCENTILE}) on {len(thr_files)}/{len(all_thr_files)} matrices.")



In [ ]:
for npz_path in thr_files:
    t0 = time.time()
    stem = Path(npz_path).stem
    out_stem = os.path.join(G_DIR, f"{stem}.thr_p{PERCENTILE}")

    sim, words = load_sim_npz(npz_path)

    edges, weights = edges_from_global_percentile(sim, p=PERCENTILE, mutual=True, weighted=True)

    n = len(words)
    m = len(edges)
    avg_degree = (2.0 * m) / max(1, n)

    g = build_igraph_threshold(n, edges, weights, words)  
    out_path = save_graph(g, out_stem)

    elapsed = time.time() - start_time

    print(f"[OK thr{PERCENTILE}] {stem}: nodes={n} edges={m} avg_deg≈{avg_degree:.1f} "
          f"→ {out_path} ({time.time()-t0:.2f}s)")